In [0]:
%sql
use catalog catalog_demos;
use schema northwind_db;

In [0]:
df = spark.table("Orders")
df.printSchema()

root
 |-- OrderID: integer (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- EmployeeID: integer (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- RequiredDate: string (nullable = true)
 |-- ShippedDate: string (nullable = true)
 |-- ShipVia: integer (nullable = true)
 |-- Freight: double (nullable = true)
 |-- ShipName: string (nullable = true)
 |-- ShipAddress: string (nullable = true)
 |-- ShipCity: string (nullable = true)
 |-- ShipRegion: string (nullable = true)
 |-- ShipPostalCode: string (nullable = true)
 |-- ShipCountry: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



## Structs
We can think of Structs as DataFrames within DataFrames

In [0]:
from pyspark.sql.functions import struct

df_structs = df.select(
    "OrderID",
    struct("ShippedDate", 
           "ShipName", 
           "ShipRegion", 
           "ShipCity").alias("ShipDetails")
)
display(df_structs.select("*").limit(2))
display(df_structs.select("OrderID", "ShipDetails.*").limit(2))
df_structs.createOrReplaceTempView("tmp_view")

OrderID,ShipDetails
10248,"List(00:00.0, Vins et alcools Chevalier, NULL, Reims)"
10249,"List(00:00.0, Toms Spezialitäten, NULL, Münster)"


OrderID,ShippedDate,ShipName,ShipRegion,ShipCity
10248,00:00.0,Vins et alcools Chevalier,NULL,Reims
10249,00:00.0,Toms Spezialitäten,NULL,Münster


In [0]:
%sql
select OrderID, ShipDetails.* from tmp_view

OrderID,ShippedDate,ShipName,ShipRegion,ShipCity
10248,00:00.0,Vins et alcools Chevalier,NULL,Reims
10249,00:00.0,Toms Spezialitäten,NULL,Münster
10250,00:00.0,Hanari Carnes,RJ,Rio de Janeiro
10251,00:00.0,Victuailles en stock,NULL,Lyon
10252,00:00.0,Suprêmes délices,NULL,Charleroi
10253,00:00.0,Hanari Carnes,RJ,Rio de Janeiro
10254,00:00.0,Chop-suey Chinese,NULL,Bern
10255,00:00.0,Richter Supermarkt,NULL,Genève
10256,00:00.0,Wellington Importadora,SP,Resende
10257,00:00.0,HILARION-Abastos,Táchira,San Cristóbal


## Arrays and explode
explode takes a column that consits of arrays and creates one row per value in the array 

In [0]:
from pyspark.sql.functions import split, explode, collect_list

df_a = spark.createDataFrame([[1, "This is a simple string"],
                              [2, "Hello World!"],], 
                             ["id", "Text"])
df_a.printSchema()
display(df_a)
df_a = df_a.withColumn("SplittedText", split("Text", " "))
display(df_a)
df_a = df_a.withColumn("ExplodedText", explode("SplittedText"))
display(df_a)
df_a = df_a.groupBy("id", "Text").agg(collect_list("ExplodedText").alias("Collected text"))
display(df_a)



root
 |-- id: long (nullable = true)
 |-- Text: string (nullable = true)



id,Text
1,This is a simple string
2,Hello World!


id,Text,SplittedText
1,This is a simple string,"List(This, is, a, simple, string)"
2,Hello World!,"List(Hello, World!)"


id,Text,SplittedText,ExplodedText
1,This is a simple string,"List(This, is, a, simple, string)",This
1,This is a simple string,"List(This, is, a, simple, string)",is
1,This is a simple string,"List(This, is, a, simple, string)",a
1,This is a simple string,"List(This, is, a, simple, string)",simple
1,This is a simple string,"List(This, is, a, simple, string)",string
2,Hello World!,"List(Hello, World!)",Hello
2,Hello World!,"List(Hello, World!)",World!


id,Text,Collected text
1,This is a simple string,"List(This, is, a, simple, string)"
2,Hello World!,"List(Hello, World!)"


## Map

In [0]:
from pyspark.sql.functions import split, explode, count, create_map, col

df_a = spark.createDataFrame([[1, "This is a simple string simple"],
                              [2, "Hello World!"]], 
                             ["id", "Text"])

# Split text into words and explode
words = df_a.withColumn("Word", explode(split("Text", " ")))

# Calculate frequency of words
freq = words.groupBy("Word").agg(count("Word").alias("cnt"))
display(freq)

# Create a map column (Word -> cnt) and explode
freq_map = freq.withColumn("Mapped", create_map(col("Word"), col("cnt")))
display(freq_map)

# Explode the map into key-value columns
exploded = freq_map.selectExpr("explode(Mapped) as (Word, cnt)")
display(exploded)



Word,cnt
string,1
is,1
a,1
This,1
simple,2
Hello,1
World!,1


Word,cnt,Mapped
string,1,Map(string -> 1)
is,1,Map(is -> 1)
a,1,Map(a -> 1)
This,1,Map(This -> 1)
simple,2,Map(simple -> 2)
Hello,1,Map(Hello -> 1)
World!,1,Map(World! -> 1)


Word,cnt
string,1
is,1
a,1
This,1
simple,2
Hello,1
World!,1
